# 从零开始手搓 DiT Diffusion Model

DiT 全名是 Diffusion Transformer。它和普通 DDPM 最大的区别不是扩散公式，而是去噪网络：

- 普通 DDPM 常用 `UNet(x_t, t)` 预测噪声。
- DiT 使用 `Transformer(x_t, t)` 预测噪声。

所以这个 notebook 会分成两层：

1. **DiT 网络层**：把图片切成 patch token，用 Transformer block 做去噪预测。
2. **DDPM 扩散层**：负责前向加噪、MSE 训练目标、反向采样公式。

你可以把 DiT 理解成：把 DDPM 里面的 UNet 换成 Vision Transformer。

In [1]:
#先实现DiT的核心逻辑和时间编码：
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def extract(v,t,x_shape):
    out = v.gather(dim=0,index=t)
    return out.float().view(t.shape[0],*((1,)*len(x_shape-1)))

def modulate(x,shift,scale):
    return x*(1+scale.unsqueeze(1)) + shift.unsqueeze(1)


def timestep_embedding(t,dim,max_period=10000):
    half = dim//2
    freqs = torch.exp(
        -math.log(max_period)*torch.arrange(0,half,deveice=t.deveice).float()/half
    )
    args = t.float()[:,None]*freqs[None]
    emb = torch.cat([torch.cos(args),torch.sin(args)],dim=-1)
    return emb

class TimeEmbedding(nn.Module):
    def __init__(self,hidden_size,frequency_embedding_size=256):
        super().__init__()
        self.frequency_emmbedding_size = frequency_embedding_size#这个参数的含义是什么？这个参数的含义是时间步嵌入的维度大小。它决定了我们在将时间步 t 转换为频率嵌入时，得到的嵌入向量的维度。通常来说，频率嵌入的维度应该与模型中其他部分使用的维度相匹配，以便后续的计算能够顺利进行。在这个代码中，频率嵌入的维度被设置为 256，但你可以根据需要进行调整。
        self.hidden_size = hidden_size
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size,hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size,hidden_size)
        )
        
    def forward(self, t):
        t_freq = timestep_embedding(t,self.frequency_emmbedding_size)
        return self.mlp(t_freq)
    

def get_1d_sincos_pos_embed(embed_dim,positions):
    assert embed_dim % 2 == 0
    omega = torch.arange(embed_dim//2).float()
    omega = omega/(embed_dim/2)
    omega = 1.0/(10000 ** omega)
    out = positions.reshape(-1)[:, None]*omega#这个代码的作用是将输入的 positions 张量进行重塑，使其成为一个列向量。具体来说，positions.reshape(-1) 会将 positions 张量展平为一个一维张量，而 [:, None] 则会在这个一维张量的基础上添加一个新的维度，使其成为一个列向量。这样做的目的是为了方便后续的计算，因为我们需要将这个列向量与 omega 张量进行逐元素相乘，以生成最终的正弦和余弦位置嵌入。
    #逐元素相乘之后形状长什么样？假设 positions 的形状是 [P]，其中 P 是位置的数量，那么 positions.reshape(-1)[:, None] 的形状将变为 [P, 1]。如果 embed_dim 是 D，那么 omega 的形状将是 [D//2]。当我们进行逐元素相乘时，out 的形状将是 [P, D//2]，因为每个位置都会与 omega 中的每个频率进行相乘，生成一个新的向量。因此，最终 out 的形状是 [P, D//2]。
    return torch.cat([torch.sin(out),torch.cos(out)],dim=1) 

def get_2d_sincos_pos_embd(embed_dim, grid_size):

    assert embed_dim % 4 == 0
    grid_h = torch.arange(grid_size).float()
    grid_w = torch.arange(grid_size).float()
    grid = torch.meshgrid(grid_h,grid_w,indexing='ij')
    grid_h, grid_w = grid[0].reshape(-1),grid[1].reshape(-1)
    emb_h = get_1d_sincos_pos_embed(embed_dim // 2,grid_h)
    emb_w = get_1d_sincos_pos_embed(embed_dim // 2,grid_w)
    return torch.cat([emb_h,emb_w],dim=1) 
#emb_h 和 emb_w 的形状是什么？假设 grid_size 是 G，那么 grid_h 和 grid_w 的形状将是 [G*G]，因为我们将二维网格展平为一个一维张量。对于 embed_dim 是 D，那么 emb_h 和 emb_w 的形状将是 [G*G, D//2]，因为每个位置都会生成一个 D//2 维的嵌入向量。因此，最终返回的张量的形状将是 [G*G, D]，因为我们将 emb_h 和 emb_w 沿着最后一个维度进行拼接。
#grid_h 和 grid_w 的形状是什么？假设 grid_size 是 G，那么 grid_h 和 grid_w 的形状将是 [G, G]，因为 torch.meshgrid 会生成一个 GxG 的网格，其中 grid_h 包含了每个位置的行索引，而 grid_w 包含了每个位置的列索引。之后，我们将它们展平为一维张量，所以最终 grid_h 和 grid_w 的形状将是 [G*G]。

class PatchEmbed(nn.Module):
    def __init__(self, image_size, patch_size, in_channels=1,hidden_szie=128):
        super().__init__()
        assert image_size%patch_size == 0
        self.img_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size//patch_size
        self.num_patch = self.grid_size**2

        self.proj = nn.Conv2d(in_channels,out_channels=hidden_szie,kernel_size=patch_size,stride=patch_size)

    def forward(self,x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1,2)

        return x
    

class MLP(nn.Module):
    def __init__(self, hidden_size, mlp_ratio = 4.0,drop = 0.0):#mlp_ratio 的含义是什么？mlp_ratio 的含义是多层感知机（MLP）中隐藏层的维度与输入维度的比例。具体来说，如果输入维度是 hidden_size，那么隐藏层的维度将是 hidden_size * mlp_ratio。这个参数控制了 MLP 中隐藏层的大小，通常来说，较大的 mlp_ratio 会增加模型的容量，但也可能导致过拟合。因此，在选择 mlp_ratio 时需要根据具体任务和数据集进行调整。
        super().__init__()
        mlp_hidden = int(hidden_size*mlp_ratio)
        self.net = nn.Sequential(
            nn.Linear(hidden_size,mlp_hidden),
            nn.GELU(approximate='tanh'),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden,hidden_size),
            nn.Dropout(drop),
        )
    
    def forward(self,x):
        return self.net(x)
    

class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size,eps=1e-6,elementwise_affine=False)
        self.attn = nn.MultiheadAttention(hidden_size,num_heads,dropout=drop,batch_first= True)
        self.norm2 = nn.LayerNorm(hidden_size,eps=1e-6,elementwise_affine=False)
        self.mlp = MLP(hidden_size,mlp_ratio,drop=drop)

        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size,6*hidden_size),
        )

    def forward(self,x,c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(c).chunk(6,dim=1)

        attn_input =  modulate(self.norm1(x),shift_msa,scale_msa)
        
        '''modulate这个函数的作用是什么?
        modulate 函数的作用是对输入张量 x 进行调制，具体来说，它会根据提供的 shift 和 scale 参数对 x 进行缩放和平移。
        函数的实现是通过将 x 乘以 (1 + scale) 来进行缩放，然后再加上 shift 来进行平移。
        这个操作可以看作是一种自适应层归一化（Adaptive Layer Normalization），它允许模型根据输入的条件 c 来动态调整特征表示，从而增强模型的表达能力和适应性。
        在 DiTBlock 中，这种调制被应用于注意力机制和 MLP 的输入，以便更好地捕捉时间步信息和图像特征之间的关系。'''
        attn_output = self.attn(attn_input,attn_input,attn_input,need_weight=False)
        #这个attn的参数传的好奇怪啊，为什么？ 这个 attn 的参数传递方式是因为 PyTorch 的 MultiheadAttention 模块需要三个输入：查询（query）、键（key）和值（value）。在这个代码中，attn_input 被同时用作查询、键和值，这是一种常见的自注意力机制的实现方式，称为“自注意力”（self-attention）。通过将同一个输入作为查询、键和值，模型能够捕捉输入序列内部的关系和依赖，从而更好地理解和处理输入数据。这种方式简化了代码，同时也符合自注意力机制的设计原则。
        x = x + gate_msa.unsqueeze(1)*attn_output
        #这个gate是用来干什么的？ 这个 gate 的作用是控制注意力输出对输入 x 的影响程度。通过将 gate_msa 扩展为与 attn_output 形状匹配的张量，并与 attn_output 逐元素相乘，我们可以动态地调整注意力输出在最终结果中的权重。这种机制允许模型根据输入的条件 c 来灵活地增强或抑制注意力输出，从而提高模型的表达能力和适应性。
        #我记得传统ViT是直接 x = x + attn_output 的，这个 gate 是 DiTBlock 中引入的一个创新点，它为模型提供了更大的灵活性，使其能够根据不同的输入条件动态调整注意力输出的影响。这种设计可以帮助模型更好地捕捉时间步信息和图像特征之间的关系，从而提升模型在处理扩散模型任务时的性能。
        mlp_input = modulate(self,self.norm2(x),shift_mlp,scale_mlp)
        mlp_output = self.mlp(mlp_input)
        mlp_output = x + mlp_output*gate_mlp.unsqueeze(1)


class FinalLayer(nn.Module):
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.final_norm = nn.LayerNorm(hidden_size,eps=1e-6,elementwise_affine=False)
        self.adaLN_modulation = nn.Sequential(
        nn.SiLU(),
        nn.Linear(hidden_size,2*hidden_size),
        )
        self.linear = nn.Linear(hidden_size,patch_size*patch_size*out_channels)

    def forward(self,x,c):
        shift,scale = self.adaLN_modulation(c).chunck(2,dim=-1)
        x = modulate(self.final_norm(x),shift,scale)
        return self.linear(x) 


class DiT(nn.Module):
    def __init__(
            self,
            image_size,
            patch_size,
            in_channels,
            hidden_size,
            depth=4,
            num_heads=4,
            mlp_ratio=4.0,
            drop=0.0
    ):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.out_channels = in_channels
        self.hidden_size = hidden_size

        self.x_embedder = PatchEmbed(image_size,patch_size,in_channels,hidden_size)
        self.t_embedder = TimeEmbedding(hidden_size)
        self.block = nn.ModuleList([
            DiTBlock(hidden_size,num_heads,mlp_ratio,drop)
            for _ in range(depth)
        ])
        self.final_layer = FinalLayer(hidden_size,patch_size,in_channels)

        pos_embed = get_2d_sincos_pos_embd(hidden_size,self.x_embedder.grid_size)
        self.register_buffer('pos_embed',pos_embed.unsqueeze(0),persistent=False)

    def unpatchify(self,x):
        B,N,patch_dim = x.shape
        p = self.patch_size
        C = self.out_channels
        grid = int(N**0.5)
        x = x.reshape(B,grid,grid,p,p,C)
        x = torch.einsum('nhwpqc->nchpwq',x)#这个函数是什么？这个函数是 PyTorch 中的 einsum 函数，它用于执行爱因斯坦求和约定的张量操作。在这个代码中，'nhwpqc->nchpwq' 是一个字符串，指定了输入张量 x 的维度标签以及输出张量的维度标签。具体来说，输入张量 x 的维度被标记为 n（批次大小）、h（网格高度）、w（网格宽度）、p（补丁大小）、q（补丁大小）和 c（通道数）。通过指定 'nhwpqc->nchpwq'，我们告诉 einsum 函数将输入张量的维度重新排列为 n（批次大小）、c（通道数）、h（网格高度）、p（补丁大小）和 w（网格宽度）。这种操作可以看作是对输入张量进行转置和重塑，以便将补丁重新组合成原始图像的形状。
        imgs = x.reshape(B,C,grid*p,grid*p)#为什么要保留C? 保留 C 是因为 C 代表了图像的通道数，例如对于 RGB 图像，C 通常是 3。通过保留 C，我们能够确保在重塑过程中正确地处理图像的颜色通道，从而得到正确的输出图像。最终输出的 imgs 张量将具有形状 [B, C, H, W]，其中 H 和 W 是根据 grid 和 patch_size 计算得出的图像高度和宽度。这种结构使得模型能够生成具有正确通道数的图像输出。
        return imgs
    
    def forward(self,x,t):
        x = self.x_embedder(x) + self.pos_embed
        c = self.t_embeedder(t)

        for block in self.block:
            x = block(x,c)
        
        x = self.final_layer(x,t)
        x = self.unpatchify(x)
        return x

In [ ]:
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


def get_device():
    """优先使用 GPU；如果 CUDA 初始化失败，就回退到 CPU。"""
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device='cuda') + 1
            return torch.device('cuda')
        except RuntimeError as err:
            print(f'CUDA 当前不可用，自动切换到 CPU：{err}')
    return torch.device('cpu')


def extract(v, t, x_shape):
    """
    DDPM 工具函数：从长度为 T 的系数表 v 中，按 batch 内每张图片的时间步 t 取系数。

    v: [T]
    t: [B]
    返回: [B, 1, 1, 1]，可以和 [B, C, H, W] 广播相乘。
    """
    out = v.gather(dim=0, index=t)
    return out.float().view(t.shape[0], *((1,) * (len(x_shape) - 1)))


def modulate(x, shift, scale):
    """
    DiT 的 AdaLN 调制：LayerNorm 后的 token 会根据时间步 t 做 shift/scale。

    x: [B, N, D]
    shift/scale: [B, D]
    返回: [B, N, D]
    """
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


device = get_device()
print('当前设备：', device)


## 1. 时间步嵌入和位置编码

DDPM 必须告诉模型当前是第几步 `t`，因为不同 `t` 的噪声强度不同。DiT 里 `t` 会变成一个向量，然后调制每个 Transformer block。

另外，Transformer 本身不知道 patch 的空间位置，所以还需要二维 sin/cos 位置编码。

In [ ]:
def timestep_embedding(t, dim, max_period=10000):
    """
    把整数时间步 t 编码成 sin/cos 向量。

    t: [B]
    dim: 输出维度
    返回: [B, dim]
    """
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(0, half, device=t.device).float() / half
    )
    args = t.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.frequency_embedding_size = frequency_embedding_size
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    def forward(self, t):
        # 先做固定 sin/cos 时间编码，再用 MLP 映射到 Transformer 的 hidden_size。
        t_freq = timestep_embedding(t, self.frequency_embedding_size)
        return self.mlp(t_freq)


def get_1d_sincos_pos_embed(embed_dim, positions):
    """一维 sin/cos 位置编码，positions shape 是 [M]。"""
    assert embed_dim % 2 == 0
    omega = torch.arange(embed_dim // 2).float()
    omega = omega / (embed_dim / 2)
    omega = 1.0 / (10000 ** omega)
    out = positions.reshape(-1)[:, None] * omega[None]
    return torch.cat([torch.sin(out), torch.cos(out)], dim=1)


def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """
    二维 sin/cos 位置编码。

    如果图片是 28x28，patch_size=4，则 grid_size=7，token 数 N=7*7=49。
    返回 shape: [N, embed_dim]
    """
    assert embed_dim % 4 == 0, '二维 sin/cos 编码要求 embed_dim 能被 4 整除'
    grid_h = torch.arange(grid_size).float()
    grid_w = torch.arange(grid_size).float()
    grid = torch.meshgrid(grid_h, grid_w, indexing='ij')
    grid_h, grid_w = grid[0].reshape(-1), grid[1].reshape(-1)
    emb_h = get_1d_sincos_pos_embed(embed_dim // 2, grid_h)
    emb_w = get_1d_sincos_pos_embed(embed_dim // 2, grid_w)
    return torch.cat([emb_h, emb_w], dim=1)


## 2. DiT 的核心模块

DiT 和 ViT 很像：先把图片 patchify 成 token，再堆 Transformer block。区别在于 DiT 不是分类，所以没有 `cls_token`；最后要把 token 还原成一张噪声图。

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, hidden_size=128):
        super().__init__()
        assert img_size % patch_size == 0, '图片尺寸必须能被 patch_size 整除'
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2

        # Conv2d 的 kernel_size=stride=patch_size，可以一步完成切 patch + 线性投影。
        self.proj = nn.Conv2d(in_channels, hidden_size, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # [B, C, H, W] -> [B, hidden_size, H/P, W/P] -> [B, N, hidden_size]
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class MLP(nn.Module):
    def __init__(self, hidden_size, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        mlp_hidden = int(hidden_size * mlp_ratio)
        self.net = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden),
            nn.GELU(approximate='tanh'),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden, hidden_size),
            nn.Dropout(drop),
        )

    def forward(self, x):
        return self.net(x)


class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()

        # elementwise_affine=False：不让 LayerNorm 自己学 shift/scale，而是交给时间条件 t 来调制。
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.mlp = MLP(hidden_size, mlp_ratio=mlp_ratio, drop=drop)

        # 一个时间向量 c 产生 6 个调制量：
        # shift/scale/gate for attention，shift/scale/gate for MLP。
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size),
        )

    def forward(self, x, c):
        # c 就是 timestep embedding。这里是 DiT 使用 DDPM 时间步 t 的关键位置。
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(c).chunk(6, dim=1)

        # 注意力分支：先 LayerNorm，再被 t 调制，然后做 self-attention。
        attn_input = modulate(self.norm1(x), shift_msa, scale_msa)
        attn_out, _ = self.attn(attn_input, attn_input, attn_input, need_weights=False)
        x = x + gate_msa.unsqueeze(1) * attn_out#为什么要加一个维度? 因为 gate_msa 是 [B, D]，而 attn_out 是 [B, N, D]，需要在 N 维上广播。
        # MLP 分支同理，也被时间步 t 调制。
        mlp_input = modulate(self.norm2(x), shift_mlp, scale_mlp)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(mlp_input)
        return x


class FinalLayer(nn.Module):
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size),
        )

        # 每个 token 要预测自己对应 patch 内每个像素的噪声。
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels)

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale)
        x = self.linear(x)
        return x


## 3. 完整 DiT 模型

`DiT.forward(x_t, t)` 的输出 shape 和输入图片一样，表示模型预测出来的噪声 `epsilon_theta(x_t, t)`。

注意：DiT 本身不负责加噪，也不负责反向采样公式；这些仍然是 DDPM 的工作。

In [ ]:
class DiT(nn.Module):
    def __init__(
        self,
        img_size=28,
        patch_size=4,
        in_channels=1,
        hidden_size=128,
        depth=4,
        num_heads=4,
        mlp_ratio=4.0,
        drop=0.0,
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.out_channels = in_channels
        self.hidden_size = hidden_size

        self.x_embedder = PatchEmbed(img_size, patch_size, in_channels, hidden_size)
        self.t_embedder = TimestepEmbedder(hidden_size)
        self.blocks = nn.ModuleList([
            DiTBlock(hidden_size, num_heads, mlp_ratio=mlp_ratio, drop=drop)
            for _ in range(depth)
        ])
        self.final_layer = FinalLayer(hidden_size, patch_size, self.out_channels)

        # DiT 论文里常用固定 2D sin/cos 位置编码，不参与训练。
        pos_embed = get_2d_sincos_pos_embed(hidden_size, self.x_embedder.grid_size)
        self.register_buffer('pos_embed', pos_embed.unsqueeze(0), persistent=False)

        self.initialize_weights()

    def initialize_weights(self):
        def basic_init(module):
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

        self.apply(basic_init)
        nn.init.xavier_uniform_(self.x_embedder.proj.weight.view(self.x_embedder.proj.weight.shape[0], -1))
        nn.init.zeros_(self.x_embedder.proj.bias)

        # AdaLN-Zero：让每个 block 一开始接近恒等映射，训练更稳。
        for block in self.blocks:
            nn.init.zeros_(block.adaLN_modulation[-1].weight)
            nn.init.zeros_(block.adaLN_modulation[-1].bias)

        # 最后一层也用零初始化，模型初始预测噪声接近 0。
        nn.init.zeros_(self.final_layer.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.final_layer.adaLN_modulation[-1].bias)
        nn.init.zeros_(self.final_layer.linear.weight)
        nn.init.zeros_(self.final_layer.linear.bias)

    def unpatchify(self, x):
        """
        token -> 图片。

        x: [B, N, patch_size * patch_size * C]
        返回: [B, C, H, W]
        """
        B, N, patch_dim = x.shape
        p = self.patch_size
        C = self.out_channels
        grid = int(N ** 0.5)
        assert grid * grid == N, 'patch 数量必须能组成正方形网格'
        assert patch_dim == p * p * C

        x = x.reshape(B, grid, grid, p, p, C)
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(B, C, grid * p, grid * p)
        return imgs

    def forward(self, x, t):
        # x 是 DDPM 当前时间步的带噪图片 x_t，t 是时间步。
        # DiT 的任务：预测这张 x_t 里面的噪声 epsilon_theta(x_t, t)。
        x = self.x_embedder(x) + self.pos_embed
        c = self.t_embedder(t)

        for block in self.blocks:
            x = block(x, c)

        x = self.final_layer(x, c)
        x = self.unpatchify(x)
        return x


## 4. DDPM 训练器：DiT 在这里作为噪声预测网络

这里就是 DiT 使用 DDPM 的第一个地方：训练时仍然按 DDPM 公式从 `x_0` 直接采样 `x_t`，然后让 DiT 预测真实噪声。

In [ ]:
class GaussianDiffusionTrainer(nn.Module):
    def __init__(self, model, beta_1=1e-4, beta_T=0.02, T=1000):
        super().__init__()
        self.model = model
        self.T = T

        self.register_buffer('betas', torch.linspace(beta_1, beta_T, T).float())
        alphas = 1.0 - self.betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        self.register_buffer('sqrt_alphas_bar', torch.sqrt(alphas_bar))
        self.register_buffer('sqrt_one_minus_alphas_bar', torch.sqrt(1.0 - alphas_bar))

    def forward(self, x_0):
        B = x_0.shape[0]
        t = torch.randint(self.T, size=(B,), device=x_0.device)
        noise = torch.randn_like(x_0)

        # DDPM 前向加噪闭式公式：x_t = sqrt(alpha_bar_t)*x_0 + sqrt(1-alpha_bar_t)*noise。
        x_t = (
            extract(self.sqrt_alphas_bar, t, x_0.shape) * x_0 +
            extract(self.sqrt_one_minus_alphas_bar, t, x_0.shape) * noise
        )

        # 这行是 DiT 和 DDPM 接起来的核心：
        # DDPM 提供 x_t 和 t；DiT 负责预测 noise。
        pred_noise = self.model(x_t, t)
        return F.mse_loss(pred_noise, noise, reduction='mean')


## 5. DDPM 采样器：反向公式不因为 DiT 改变

这里是 DiT 使用 DDPM 的第二个地方：采样时仍然从纯噪声 `x_T` 开始，用 DDPM 的反向公式一步步得到 `x_0`。DiT 只在每一步预测 `epsilon_theta(x_t, t)`。

In [ ]:
class GaussianDiffusionSampler(nn.Module):
    def __init__(self, model, beta_1=1e-4, beta_T=0.02, T=1000):
        super().__init__()
        self.model = model
        self.T = T

        self.register_buffer('betas', torch.linspace(beta_1, beta_T, T).float())
        alphas = 1.0 - self.betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        alphas_bar_prev = F.pad(alphas_bar, [1, 0], value=1.0)[:T]

        self.register_buffer('coeff1', torch.sqrt(1.0 / alphas))
        self.register_buffer('coeff2', self.coeff1 * (1.0 - alphas) / torch.sqrt(1.0 - alphas_bar))
        self.register_buffer('posterior_var', self.betas * (1.0 - alphas_bar_prev) / (1.0 - alphas_bar))

    def predict_xt_prev_mean_from_eps(self, x_t, t, eps):
        return (
            extract(self.coeff1, t, x_t.shape) * x_t -
            extract(self.coeff2, t, x_t.shape) * eps
        )

    def p_mean_variance(self, x_t, t):
        var = torch.cat([self.posterior_var[1:2], self.betas[1:]])
        var = extract(var, t, x_t.shape)

        # DDPM 每个反向步都需要一个噪声预测器；这里的预测器就是 DiT。
        eps = self.model(x_t, t)
        xt_prev_mean = self.predict_xt_prev_mean_from_eps(x_t, t, eps=eps)
        return xt_prev_mean, var

    @torch.no_grad()
    def forward(self, x_T):
        x_t = x_T
        for time_step in reversed(range(self.T)):
            t = x_t.new_full((x_t.shape[0],), time_step, dtype=torch.long)
            mean, var = self.p_mean_variance(x_t, t)

            if time_step > 0:
                noise = torch.randn_like(x_t)
            else:
                noise = 0

            x_t = mean + torch.sqrt(var) * noise
            assert torch.isnan(x_t).int().sum() == 0, '采样过程中出现 NaN'

        return torch.clip(x_t, -1.0, 1.0)


## 6. 冒烟测试

先跑一个很小的 DiT，确认 shape、loss、采样都能通。

In [ ]:
# 冒烟测试只检查代码能不能跑通，所以 T 用很小的 5。
# 正式训练时可以在下一节使用 T=200 或 T=1000。
T = 5
model = DiT(
    img_size=28,
    patch_size=4,
    in_channels=1,
    hidden_size=64,
    depth=2,
    num_heads=4,
).to(device)
trainer = GaussianDiffusionTrainer(model, T=T).to(device)
sampler = GaussianDiffusionSampler(model, T=T).to(device)

x = torch.randn(2, 1, 28, 28, device=device)
t = torch.randint(T, (2,), device=device)

with torch.no_grad():
    pred_noise = model(x, t)
    loss = trainer(x)
    sample = sampler(torch.randn(2, 1, 28, 28, device=device))

print('DiT 输出 shape:', pred_noise.shape)
print('训练 loss:', float(loss))
print('采样输出 shape:', sample.shape)


## 7. MNIST 训练入口

第一次建议先用 `T=200`、`hidden_size=128` 跑通。想更接近标准 DDPM，可以把 `T` 改成 1000，但训练和采样都会更慢。

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

data_dir = Path('./data')
train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)

# 快速实验可以打开这一行，只用一小部分 MNIST。
# train_dataset = Subset(train_dataset, range(4096))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=(device.type == 'cuda'))

T = 200
model = DiT(img_size=28, patch_size=4, in_channels=1, hidden_size=128, depth=4, num_heads=4).to(device)
trainer = GaussianDiffusionTrainer(model, beta_1=1e-4, beta_T=0.02, T=T).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

epochs = 1
model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    total_images = 0

    for step, (images, _) in enumerate(train_loader, start=1):
        images = images.to(device)
        optimizer.zero_grad()
        loss = trainer(images)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_images += images.size(0)

        if step % 100 == 0:
            print(f'Epoch {epoch:02d} | step {step:04d} | loss={loss.item():.4f}')

    print(f'Epoch {epoch:02d} 完成 | avg_loss={total_loss / total_images:.4f}')

torch.save(model.state_dict(), 'dit_mnist.pth')
print('模型权重已保存到 dit_mnist.pth')


## 8. 采样可视化

训练结束后运行这一格，从随机噪声生成 MNIST 风格图片。训练轮数少时结果会比较糊，这是正常的。

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid


# 如果已经保存过权重，可以取消注释直接加载。
# model.load_state_dict(torch.load('dit_mnist.pth', map_location=device))
# model.to(device)

model.eval()
sampler = GaussianDiffusionSampler(model, beta_1=1e-4, beta_T=0.02, T=T).to(device)

x_T = torch.randn(16, 1, 28, 28, device=device)
samples = sampler(x_T)
samples = (samples + 1) / 2

grid = make_grid(samples, nrow=4).permute(1, 2, 0).cpu().numpy()
plt.figure(figsize=(5, 5))
plt.imshow(grid, cmap='gray')
plt.axis('off')
plt.show()
